## Reduce Complexity of Model
As we've seen in creating Supervised and Unsupervised Machine Learning models, the number of predictors plays a large role in each model. The more predictors we have, the more complex the model and the higher risk of overfitting our training data in a way that is not predictive. So the major question is: how do we reduce the complexity of our model without indescriminantly throwing out data? 

## Feature Selection or Feature Projection?

There are two general methods to reducing the number for features in your model: selection and projection. 

**Feature Selection** involves systematically choosing a smaller set of features to focus on and removing the rest. For example, consider our 13-feature wine dataset. One could systematically create Decision Tree models for every possible 4-feature combination *(715 combinations in total)* and compare the resulting five-fold cross-validation accuracy averages to determine the 4 features to keep. 

**Feature Projection** involves mathematically combining multiple features into new, singular "Frankenstein" features. Principal Component Analysis (PCA) is such a method, where the principal axes are linear combinations of multiple features. This still results in loss of data, but in a "mathematically minimal" way. Explaining the mathematics behind these methods is beyond the scope of this survey course and is best revisited after taking a Linear Algebra course.

## Feature Selection Example - Combinatorial Optimization
Let's write out the code to reduce our 13-feature wine dataset down to 2 features. 

In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier

from itertools import combinations
from sklearn.model_selection import cross_val_score

In [3]:
def create_all_feature_dicts(df, subset_n):
    features = df.keys()
    all_subsets = list(combinations(features, subset_n))

    list_of_feature_dicts = []
    name_index = 0
    for subset in all_subsets:
        temp_dict = {
            'name': f'model_{name_index}',
            'features': list(subset), 
            'reduced_df': df[list(subset)], 
            'target': wine.target,
            'model': DecisionTreeClassifier()
        }
        list_of_feature_dicts.append(temp_dict)
        name_index += 1

    return list_of_feature_dicts

def run_cross_val(dict, cv_n):
    model = dict['model']
    X = dict['reduced_df']
    y = dict['target']
    all_scores = cross_val_score(model, X, y, cv=cv_n)

    dict['cross_val_scores'] = all_scores
    dict['cross_val_avg'] = np.mean(all_scores)
    return dict

def return_selected_dict(list_of_dicts, name):
    selected_dict = list(filter(lambda model: model['name'] == name, list_of_dicts))[0]
    return selected_dict

In [4]:
wine = load_wine()
list_of_feature_dicts = create_all_feature_dicts(pd.DataFrame(wine.data), 2)

list_of_cross_val_avg = []
list_of_model_names = []

for dict in list_of_feature_dicts:
    dict = run_cross_val(dict, 5)
    list_of_cross_val_avg.append(dict['cross_val_avg'])
    list_of_model_names.append(dict['name'])

sorted_model_names = [name for _, name in sorted(zip(list_of_cross_val_avg, list_of_model_names), reverse=True)]

for index in range(0, 10):
    model_name = sorted_model_names[index]
    model_dict = return_selected_dict(list_of_feature_dicts, model_name)
    print(f'Rank: {index+1}')
    print(f'Features: {model_dict['features']}')
    print(f'Cross-Validation Average: {int(model_dict['cross_val_avg']*100)}%')
    print()

Rank: 1
Features: [6, 9]
Cross-Validation Average: 91%

Rank: 2
Features: [0, 6]
Cross-Validation Average: 88%

Rank: 3
Features: [9, 12]
Cross-Validation Average: 87%

Rank: 4
Features: [5, 9]
Cross-Validation Average: 87%

Rank: 5
Features: [9, 11]
Cross-Validation Average: 86%

Rank: 6
Features: [10, 12]
Cross-Validation Average: 86%

Rank: 7
Features: [0, 11]
Cross-Validation Average: 84%

Rank: 8
Features: [4, 6]
Cross-Validation Average: 83%

Rank: 9
Features: [11, 12]
Cross-Validation Average: 83%

Rank: 10
Features: [8, 9]
Cross-Validation Average: 82%



## Feature Projection Example - Linear Discriminant Analysis (LDA)

IBM hosts an *excellent* step-by-step description of Linear Discriminant Analysis (LDA): [https://developer.ibm.com/tutorials/awb-implementing-linear-discriminant-analysis-python/](https://developer.ibm.com/tutorials/awb-implementing-linear-discriminant-analysis-python/). Advanced users looking to see how to determine how many features to project onto should read through this resource.

In [ ]:
import numpy as np

from sklearn.datasets import load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

wine = load_wine()
X = wine.data
y = wine.target

# Apply Linear Discriminant Analysis
# n_components cannot be more than min(n_features, n_classes - 1)
# Since n_classes = 3, our n_components is necessarily = 2
lda = LinearDiscriminantAnalysis(n_components=2) 

# Explicity write out our 5-fold cross validation sets
# Stratified means we are retaining percentage of classes in each subset 
skf = StratifiedKFold(n_splits=5, shuffle=True)

list_of_accuracy_scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train = lda.fit_transform(X_train, y_train)
    X_test = lda.transform(X_test)

    fold_model = RandomForestClassifier()

    fold_model.fit(X_train, y_train)

    predictions = fold_model.predict(X_test)

    # Evaluate accuracy
    accuracy = accuracy_score(y_test, predictions)
    list_of_accuracy_scores.append(accuracy)

avg_accuracy = np.mean(list_of_accuracy_scores)
print(f'Average Accuracy for 5-fold cross-validation after LDA: {int(avg_accuracy*100)}%')

Average Accuracy for 5-fold cross-validation after LDA: 99%


## Feature Projection Example - Principal Component Analysis (PCA)

### Functions to Graph SVCs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# import the functions needed from sklearn
from sklearn.datasets import load_wine
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def make_meshgrid(x, y, h=.05):
    x_min, x_max = x.min() - 1, x.max() + 1
    y_min, y_max = y.min() - 1, y.max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    return xx, yy

def plot_contours(ax, clf, xx, yy, **params):
    print('This may take 30-60 seconds to run. Please be patient.')
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    out = ax.contourf(xx, yy, Z, **params)
    return out

def create_graph(clf_2feature, feature_names, X_train, y_train):
    fig, ax = plt.subplots()
    # title for the plots
    title = ('Decision surface of linear SVC ')
    # Set-up grid for plotting
    X0, X1 = X_train[:, 0], X_train[:, 1]
    xx, yy = make_meshgrid(X0, X1)

    plot_contours(ax, clf_2feature, xx, yy, cmap=plt.cm.coolwarm, alpha=0.8)
    ax.scatter(X0, X1, c=y_train, cmap=plt.cm.coolwarm, s=20, edgecolors='k')
    ax.set_ylabel(f'Wine feature: {feature_names[1]}')
    ax.set_xlabel(f'Wine feature: {feature_names[0]}')
    ax.set_xticks(())
    ax.set_yticks(())
    ax.set_title(title)
    #ax.legend()

    plt.show()

def run_svm(data, target, type):
    X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2)

    model = SVC(kernel='linear')
    clf = model.fit(X_train, y_train)

    # Make predictions
    predictions = clf.predict(X_test)

    # Evaluate accuracy
    accuracy = accuracy_score(y_test, predictions)
    print(f'SVM Accuracy on {type} Data: {round(accuracy, 3)*100}%')

    return [clf, X_train, y_train]

### Running SVC on PCA-Transformed Data vs non-PCA-Transformed Data

In [ ]:
wine = load_wine()

pca_2 = PCA(n_components=2)
pca_transformed_data = pca_2.fit_transform(wine.data)
pca_feature_names = ['pc_0', 'pc_1']

clf_base, X_train_base, y_train_base = run_svm(wine.data[:, :2], wine.target, 'Untransformed')
clf_trans, X_train_trans, y_train_trans = run_svm(pca_transformed_data, wine.target, 'PCA-Transformed')

In [ ]:
create_graph(clf_base, wine.feature_names[0:2], X_train_base, y_train_base)

In [ ]:
create_graph(clf_trans, pca_feature_names, X_train_trans, y_train_trans)

Notice how the boundaries change between the two models since the data is transformed. **This may or may not improve the accuracy of your model!** 